# Dynamic Programming in ML

DP appears throughout ML: structured prediction, sequence decoding, alignment.

Covers:
1. Viterbi algorithm (general trellis formulation)
2. CTC decoding (greedy + beam search)
3. Edit distance / Levenshtein distance
4. Sequence alignment (Needleman-Wunsch)
5. Beam search as DP (general formulation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)

## 1. Viterbi Algorithm

**Problem**: Find the most probable sequence of hidden states given observations.

**General formulation** (not HMM-specific):
- States: $S = \{s_1, \ldots, s_K\}$
- Observations: $O = (o_1, \ldots, o_T)$
- Transition scores: $A(s_i, s_j)$ (score of transitioning from state $i$ to state $j$)
- Emission scores: $B(s_i, o_t)$ (score of emitting $o_t$ in state $s_i$)
- Initial scores: $\pi(s_i)$

**DP recurrence** (log-space):
$$V_t(j) = B(s_j, o_t) + \max_i [V_{t-1}(i) + A(s_i, s_j)]$$

**Complexity**: $O(T \cdot K^2)$ -- linear in sequence length, quadratic in number of states.

In [ ]:
def viterbi(log_initial, log_transition, log_emission):
    """
    Viterbi algorithm for finding the most likely state sequence.
    
    log_initial: (K,) log initial state probabilities
    log_transition: (K, K) log_transition[i,j] = log P(state_j | state_i)
    log_emission: (T, K) log_emission[t,k] = log P(obs_t | state_k)
    
    Returns: best_path (T,), best_score
    """
    T, K = log_emission.shape
    
    # Trellis: V[t, k] = best log-score ending in state k at time t
    V = np.full((T, K), -np.inf)
    backpointer = np.zeros((T, K), dtype=int)
    
    # Initialization
    V[0] = log_initial + log_emission[0]
    
    # Forward pass: fill trellis
    for t in range(1, T):
        for j in range(K):
            # Best previous state transitioning to j
            scores = V[t-1] + log_transition[:, j]  # (K,)
            best_prev = np.argmax(scores)
            V[t, j] = scores[best_prev] + log_emission[t, j]
            backpointer[t, j] = best_prev
    
    # Backtrack: find best ending state, then trace back
    best_path = np.zeros(T, dtype=int)
    best_path[T-1] = np.argmax(V[T-1])
    best_score = V[T-1, best_path[T-1]]
    
    for t in range(T-2, -1, -1):
        best_path[t] = backpointer[t+1, best_path[t+1]]
    
    return best_path, best_score, V, backpointer


# Example: Weather HMM (classic textbook example)
# Hidden states: Sunny(0), Rainy(1)
# Observations: Walk(0), Shop(1), Clean(2)
states = ['Sunny', 'Rainy']
observations_vocab = ['Walk', 'Shop', 'Clean']

log_initial = np.log([0.6, 0.4])  # P(start)
log_transition = np.log([[0.7, 0.3],   # P(next | Sunny)
                         [0.4, 0.6]])   # P(next | Rainy)

emission_probs = np.array([[0.6, 0.3, 0.1],   # P(obs | Sunny)
                           [0.1, 0.4, 0.5]])   # P(obs | Rainy)

# Observation sequence: Walk, Shop, Clean, Clean, Walk
obs_sequence = [0, 1, 2, 2, 0]
T = len(obs_sequence)
log_emission = np.log(emission_probs[:, obs_sequence].T)  # (T, K)

best_path, best_score, trellis, bp = viterbi(log_initial, log_transition, log_emission)

print("Observations:", [observations_vocab[o] for o in obs_sequence])
print("Best path:   ", [states[s] for s in best_path])
print(f"Log score:    {best_score:.4f}")

In [ ]:
# Visualize the Viterbi trellis
fig, ax = plt.subplots(figsize=(12, 5))

K = len(states)
# Draw trellis nodes
for t in range(T):
    for k in range(K):
        color = 'gold' if best_path[t] == k else 'lightblue'
        circle = plt.Circle((t, k), 0.15, color=color, ec='black', zorder=5)
        ax.add_patch(circle)
        ax.text(t, k, f'{np.exp(trellis[t,k]):.4f}', ha='center', va='center', fontsize=7, zorder=6)

# Draw all transition arrows (light)
for t in range(1, T):
    for k_to in range(K):
        for k_from in range(K):
            ax.annotate('', xy=(t - 0.15, k_to), xytext=(t - 1 + 0.15, k_from),
                       arrowprops=dict(arrowstyle='->', color='lightgray', lw=0.5))

# Draw best path arrows (bold)
for t in range(1, T):
    ax.annotate('', xy=(t - 0.15, best_path[t]),
               xytext=(t - 1 + 0.15, best_path[t-1]),
               arrowprops=dict(arrowstyle='->', color='red', lw=2.5))

ax.set_xlim(-0.5, T - 0.5)
ax.set_ylim(-0.5, K - 0.5)
ax.set_xticks(range(T))
ax.set_xticklabels([observations_vocab[o] for o in obs_sequence])
ax.set_yticks(range(K))
ax.set_yticklabels(states)
ax.set_xlabel('Observations')
ax.set_ylabel('Hidden States')
ax.set_title('Viterbi Trellis (red = best path, values = path probabilities)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 2. CTC Decoding

**Connectionist Temporal Classification** solves the alignment problem in sequence-to-sequence tasks where input and output lengths differ (e.g., speech recognition, OCR).

**Why CTC is needed**: The model outputs a probability distribution over labels at each input frame, but we do not know the alignment between frames and output tokens. CTC introduces a blank token and a many-to-one mapping:
- Collapse repeated characters: `aaa` -> `a`
- Remove blank tokens: `a-b--b` -> `abb` (where `-` is blank)

This allows many valid alignments for the same output.

In [ ]:
def ctc_greedy_decode(log_probs, blank=0):
    """
    Greedy CTC decoding: take argmax at each timestep, then collapse.
    
    log_probs: (T, V) log probabilities over vocabulary at each timestep
    blank: index of the blank token
    
    Returns: decoded sequence (list of token indices, excluding blank)
    """
    T, V = log_probs.shape
    
    # Step 1: argmax at each timestep
    best_path = np.argmax(log_probs, axis=1)  # (T,)
    
    # Step 2: collapse repeated characters and remove blanks
    decoded = []
    prev = None
    for token in best_path:
        if token != prev:  # collapse repeats
            if token != blank:  # skip blank
                decoded.append(token)
        prev = token
    
    return decoded, best_path


# Generate synthetic CTC output (e.g., recognizing "hello")
# Vocabulary: 0=blank, 1=h, 2=e, 3=l, 4=o
vocab = {0: '-', 1: 'h', 2: 'e', 3: 'l', 4: 'o'}
V = len(vocab)
T_ctc = 12  # input frames (longer than output "hello" = 5 chars)

# Simulate model output: spiky probabilities at correct positions
# True alignment: --h-ee-ll-oo
true_alignment = [0, 0, 1, 0, 2, 2, 0, 3, 3, 0, 4, 4]
log_probs = np.full((T_ctc, V), -5.0)  # low baseline
for t, label in enumerate(true_alignment):
    log_probs[t, label] = -0.1  # high probability for correct label
    # Add some noise
    log_probs[t] += np.random.randn(V) * 0.3

# Normalize to valid log-probs
log_probs -= np.log(np.sum(np.exp(log_probs), axis=1, keepdims=True))

decoded, raw_path = ctc_greedy_decode(log_probs, blank=0)
print("Raw path:   ", ''.join([vocab[t] for t in raw_path]))
print("Decoded:    ", ''.join([vocab[t] for t in decoded]))
print("Expected:    hello")

In [ ]:
def ctc_beam_search(log_probs, beam_width=5, blank=0):
    """
    CTC beam search decoding with prefix merging.
    
    Key idea: Track two scores for each prefix:
    - p_b: probability of paths ending in blank
    - p_nb: probability of paths not ending in blank
    This handles the CTC collapsing rule correctly.
    
    log_probs: (T, V) log probabilities
    Returns: best decoded sequence
    """
    T, V = log_probs.shape
    NEG_INF = -float('inf')
    
    # Each beam entry: prefix -> (log_p_blank, log_p_non_blank)
    # Use tuples as keys for prefixes
    beams = {(): (0.0, NEG_INF)}  # empty prefix starts with blank probability = 1
    
    def log_add(a, b):
        """Numerically stable log(exp(a) + exp(b))."""
        if a == NEG_INF:
            return b
        if b == NEG_INF:
            return a
        if a > b:
            return a + np.log1p(np.exp(b - a))
        return b + np.log1p(np.exp(a - b))
    
    for t in range(T):
        new_beams = defaultdict(lambda: (NEG_INF, NEG_INF))
        
        # Prune to top beam_width entries
        scored = [(prefix, log_add(p_b, p_nb)) for prefix, (p_b, p_nb) in beams.items()]
        scored.sort(key=lambda x: x[1], reverse=True)
        scored = scored[:beam_width]
        
        for prefix, _ in scored:
            p_b, p_nb = beams[prefix]
            p_total = log_add(p_b, p_nb)
            
            for c in range(V):
                lp = log_probs[t, c]
                
                if c == blank:
                    # Extending with blank: merge into same prefix
                    old_b, old_nb = new_beams[prefix]
                    new_beams[prefix] = (log_add(old_b, p_total + lp), old_nb)
                else:
                    # Extending with non-blank character
                    if len(prefix) > 0 and prefix[-1] == c:
                        # Same char as end of prefix:
                        # If previous was blank, we can extend (new char)
                        # If previous was not blank, it's a repeat (collapse)
                        new_prefix = prefix + (c,)
                        old_b, old_nb = new_beams[new_prefix]
                        new_beams[new_prefix] = (old_b, log_add(old_nb, p_b + lp))
                        
                        # Also: repeat same char (stays as same prefix)
                        old_b2, old_nb2 = new_beams[prefix]
                        new_beams[prefix] = (old_b2, log_add(old_nb2, p_nb + lp))
                    else:
                        # Different char: extend prefix
                        new_prefix = prefix + (c,)
                        old_b, old_nb = new_beams[new_prefix]
                        new_beams[new_prefix] = (old_b, log_add(old_nb, p_total + lp))
        
        beams = dict(new_beams)
    
    # Find best prefix
    best_prefix = max(beams.keys(), key=lambda p: log_add(beams[p][0], beams[p][1]))
    best_score = log_add(beams[best_prefix][0], beams[best_prefix][1])
    
    return list(best_prefix), best_score, beams


# Test beam search
beam_decoded, beam_score, final_beams = ctc_beam_search(log_probs, beam_width=10, blank=0)
print("Beam search decoded:", ''.join([vocab[t] for t in beam_decoded]))
print(f"Beam search score: {beam_score:.4f}")

# Show top beams
def log_add_safe(a, b):
    if a == -float('inf'): return b
    if b == -float('inf'): return a
    return max(a, b) + np.log1p(np.exp(-abs(a - b)))

print("\nTop 5 beam candidates:")
scored_beams = [(p, log_add_safe(pb, pnb)) for p, (pb, pnb) in final_beams.items()]
scored_beams.sort(key=lambda x: x[1], reverse=True)
for prefix, score in scored_beams[:5]:
    text = ''.join([vocab[t] for t in prefix])
    print(f"  '{text}' (score={score:.4f})")

## 3. Edit Distance (Levenshtein Distance)

**Problem**: Minimum number of single-character edits (insert, delete, substitute) to transform string A into string B.

**DP recurrence**:
$$D(i, j) = \begin{cases}
i & \text{if } j = 0 \\
j & \text{if } i = 0 \\
D(i-1, j-1) & \text{if } a_i = b_j \\
1 + \min\{D(i-1,j), D(i,j-1), D(i-1,j-1)\} & \text{otherwise}
\end{cases}$$

**Complexity**: $O(mn)$ time and space (or $O(\min(m,n))$ space with optimization).

In [ ]:
def edit_distance(a, b, return_table=False):
    """
    Compute Levenshtein distance between strings a and b.
    Returns distance and optionally the full DP table.
    """
    m, n = len(a), len(b)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    
    # Base cases
    for i in range(m + 1):
        dp[i, 0] = i
    for j in range(n + 1):
        dp[0, j] = j
    
    # Fill table
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i, j] = dp[i-1, j-1]  # match
            else:
                dp[i, j] = 1 + min(
                    dp[i-1, j],    # delete from a
                    dp[i, j-1],    # insert into a
                    dp[i-1, j-1]   # substitute
                )
    
    if return_table:
        return dp[m, n], dp
    return dp[m, n]


# Demo
a = "kitten"
b = "sitting"
dist, table = edit_distance(a, b, return_table=True)
print(f"edit_distance('{a}', '{b}') = {dist}")
print(f"Operations: k->s, e->i, +g (3 edits)")

In [ ]:
# Visualize DP table
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(table, cmap='YlOrRd', aspect='auto')

# Add text annotations
for i in range(table.shape[0]):
    for j in range(table.shape[1]):
        ax.text(j, i, str(table[i, j]), ha='center', va='center', fontsize=10)

# Labels
ax.set_xticks(range(len(b) + 1))
ax.set_xticklabels([''] + list(b))
ax.set_yticks(range(len(a) + 1))
ax.set_yticklabels([''] + list(a))
ax.set_xlabel(f"Target: '{b}'")
ax.set_ylabel(f"Source: '{a}'")
ax.set_title(f"Edit Distance DP Table (distance = {dist})")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 4. Sequence Alignment (Needleman-Wunsch)

**Problem**: Find the optimal global alignment between two sequences, allowing gaps.

Similar to edit distance but with a scoring scheme:
- Match: +1 (or from substitution matrix)
- Mismatch: -1
- Gap: -2 (penalty for insertions/deletions)

**DP recurrence**:
$$F(i,j) = \max\begin{cases}
F(i-1, j-1) + s(a_i, b_j) & \text{(match/mismatch)} \\
F(i-1, j) + g & \text{(gap in b)} \\
F(i, j-1) + g & \text{(gap in a)}
\end{cases}$$

In [ ]:
def needleman_wunsch(seq_a, seq_b, match=1, mismatch=-1, gap=-2):
    """
    Global sequence alignment using Needleman-Wunsch algorithm.
    Returns: score, aligned_a, aligned_b, score_matrix
    """
    m, n = len(seq_a), len(seq_b)
    
    # Score matrix
    F = np.zeros((m + 1, n + 1))
    traceback = np.zeros((m + 1, n + 1), dtype=int)  # 0=diag, 1=up, 2=left
    
    # Initialize
    for i in range(1, m + 1):
        F[i, 0] = i * gap
        traceback[i, 0] = 1  # up
    for j in range(1, n + 1):
        F[0, j] = j * gap
        traceback[0, j] = 2  # left
    
    # Fill
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = match if seq_a[i-1] == seq_b[j-1] else mismatch
            scores = [
                F[i-1, j-1] + s,  # diagonal (match/mismatch)
                F[i-1, j] + gap,  # up (gap in b)
                F[i, j-1] + gap,  # left (gap in a)
            ]
            best = np.argmax(scores)
            F[i, j] = scores[best]
            traceback[i, j] = best
    
    # Traceback
    aligned_a, aligned_b = [], []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and traceback[i, j] == 0:
            aligned_a.append(seq_a[i-1])
            aligned_b.append(seq_b[j-1])
            i -= 1
            j -= 1
        elif i > 0 and traceback[i, j] == 1:
            aligned_a.append(seq_a[i-1])
            aligned_b.append('-')
            i -= 1
        else:
            aligned_a.append('-')
            aligned_b.append(seq_b[j-1])
            j -= 1
    
    aligned_a.reverse()
    aligned_b.reverse()
    
    return F[m, n], ''.join(aligned_a), ''.join(aligned_b), F


# Demo: protein-like sequence alignment
seq_a = "GATTACA"
seq_b = "GCATGCU"

score, align_a, align_b, score_matrix = needleman_wunsch(seq_a, seq_b)

print(f"Sequence A: {seq_a}")
print(f"Sequence B: {seq_b}")
print(f"\nOptimal alignment (score = {score:.0f}):")
print(f"  {align_a}")
# Show match indicators
match_str = ''.join(['|' if a == b else ' ' for a, b in zip(align_a, align_b)])
print(f"  {match_str}")
print(f"  {align_b}")

In [ ]:
# Visualize alignment matrix
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(score_matrix, cmap='RdYlGn', aspect='auto')

for i in range(score_matrix.shape[0]):
    for j in range(score_matrix.shape[1]):
        ax.text(j, i, f'{score_matrix[i,j]:.0f}', ha='center', va='center', fontsize=9)

ax.set_xticks(range(len(seq_b) + 1))
ax.set_xticklabels([''] + list(seq_b))
ax.set_yticks(range(len(seq_a) + 1))
ax.set_yticklabels([''] + list(seq_a))
ax.set_xlabel(f"Sequence B: {seq_b}")
ax.set_ylabel(f"Sequence A: {seq_a}")
ax.set_title(f"Needleman-Wunsch Score Matrix (score = {score:.0f})")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 5. Beam Search as DP

**General formulation**: At each step, maintain the top-k partial hypotheses ("beams"). Extend each by all possible next tokens, score, and prune back to k.

This is DP with pruning: instead of keeping all states, we keep only the top-k.

**Complexity**: $O(T \cdot k \cdot V)$ where T = steps, k = beam width, V = vocabulary size.

**Beyond LLM decoding**: Beam search applies to any sequential decision problem -- machine translation, speech recognition, planning, etc.

In [ ]:
def beam_search(score_fn, vocab_size, max_len, beam_width, eos_token=None):
    """
    General beam search.
    
    score_fn(prefix) -> log_probs: (V,) array of log probabilities for next token
    vocab_size: number of possible tokens
    max_len: maximum sequence length
    beam_width: number of hypotheses to keep
    eos_token: if set, completed sequences are stored separately
    
    Returns: list of (sequence, score) tuples
    """
    # Each beam: (log_score, prefix)
    beams = [(0.0, [])]
    completed = []
    
    for step in range(max_len):
        candidates = []
        
        for score, prefix in beams:
            # Get next token scores
            next_scores = score_fn(prefix)  # (V,)
            
            for token in range(vocab_size):
                new_score = score + next_scores[token]
                new_prefix = prefix + [token]
                
                if eos_token is not None and token == eos_token:
                    completed.append((new_score, new_prefix))
                else:
                    candidates.append((new_score, new_prefix))
        
        # Prune to top beam_width
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]
        
        if len(beams) == 0:
            break
    
    # Combine completed and remaining beams
    all_results = completed + beams
    all_results.sort(key=lambda x: x[0], reverse=True)
    
    return all_results


# Demo: simple language model decoding
# Vocabulary: 0=<eos>, 1=the, 2=cat, 3=sat, 4=on, 5=mat, 6=dog, 7=ran
vocab_lm = {0: '<eos>', 1: 'the', 2: 'cat', 3: 'sat', 4: 'on', 5: 'mat', 6: 'dog', 7: 'ran'}
V_lm = len(vocab_lm)

# Fake "language model": assigns higher scores to reasonable sequences
# This is a simplified bigram-like model
bigram_scores = np.full((V_lm, V_lm), -3.0)  # default: unlikely
# Likely transitions
bigram_scores[1, 2] = -0.5  # the -> cat
bigram_scores[1, 6] = -0.7  # the -> dog
bigram_scores[2, 3] = -0.3  # cat -> sat
bigram_scores[6, 7] = -0.4  # dog -> ran
bigram_scores[3, 4] = -0.3  # sat -> on
bigram_scores[4, 1] = -0.3  # on -> the
bigram_scores[1, 5] = -0.5  # the -> mat
bigram_scores[5, 0] = -0.1  # mat -> eos
bigram_scores[7, 0] = -0.2  # ran -> eos

# Normalize each row to be valid log-probs
for i in range(V_lm):
    bigram_scores[i] -= np.log(np.sum(np.exp(bigram_scores[i])))

# Start token distribution
start_scores = np.full(V_lm, -5.0)
start_scores[1] = -0.1  # start with "the"
start_scores -= np.log(np.sum(np.exp(start_scores)))

def fake_lm(prefix):
    if len(prefix) == 0:
        return start_scores
    return bigram_scores[prefix[-1]]


# Run beam search
results = beam_search(fake_lm, V_lm, max_len=7, beam_width=5, eos_token=0)

print("Beam search results (top 10):")
for score, seq in results[:10]:
    text = ' '.join([vocab_lm[t] for t in seq])
    print(f"  score={score:.3f}  {text}")

In [ ]:
# Visualize beam search tree for first few steps
def visualize_beam_tree(score_fn, vocab, beam_width=3, max_depth=4):
    """Visualize the beam search expansion as a tree."""
    fig, ax = plt.subplots(figsize=(14, 8))
    
    beams = [(0.0, [])]
    y_positions = {tuple(): 0}
    node_count = 0
    
    all_nodes = [(0, 0.0, '', None)]  # (depth, score, label, parent_key)
    node_keys = {tuple(): 0}
    
    for step in range(max_depth):
        candidates = []
        for score, prefix in beams:
            next_scores = score_fn(prefix)
            top_k = np.argsort(next_scores)[-beam_width:][::-1]
            for token in top_k:
                new_score = score + next_scores[token]
                new_prefix = prefix + [token]
                candidates.append((new_score, new_prefix))
                node_count += 1
                key = tuple(new_prefix)
                node_keys[key] = node_count
                all_nodes.append((step + 1, new_score, vocab[token], tuple(prefix)))
        
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]
    
    # Layout: assign y positions per depth level
    depths = defaultdict(list)
    for i, (depth, score, label, parent) in enumerate(all_nodes):
        depths[depth].append(i)
    
    positions = {}
    for depth, indices in depths.items():
        n = len(indices)
        for j, idx in enumerate(indices):
            positions[idx] = (depth, j - n / 2)
    
    # Draw edges and nodes
    for i, (depth, score, label, parent_key) in enumerate(all_nodes):
        x, y_pos = positions[i]
        
        if parent_key is not None and parent_key in node_keys:
            parent_idx = node_keys[parent_key]
            if parent_idx in positions:
                px, py = positions[parent_idx]
                ax.plot([px, x], [py, y_pos], 'gray', alpha=0.3, linewidth=1)
        
        # Check if this node is in the final beam
        is_final = any(tuple(prefix) in node_keys and node_keys[tuple(prefix)] == i 
                       for _, prefix in beams) if depth == max_depth else False
        color = 'gold' if is_final else 'lightblue'
        ax.scatter(x, y_pos, s=200, c=color, edgecolors='black', zorder=5)
        ax.text(x, y_pos, f'{label}\n{score:.1f}', ha='center', va='center', fontsize=6, zorder=6)
    
    ax.set_xlabel('Decoding Step')
    ax.set_title(f'Beam Search Tree (beam_width={beam_width})')
    ax.set_xticks(range(max_depth + 1))
    plt.tight_layout()
    plt.show()

visualize_beam_tree(fake_lm, vocab_lm, beam_width=3, max_depth=4)

## 6. CTC Decoding Demo: Sequence Labeling

Simulate a model recognizing the sequence "abc" from 8 input frames.

In [ ]:
# Vocabulary: 0=blank, 1=a, 2=b, 3=c
ctc_vocab = {0: '_', 1: 'a', 2: 'b', 3: 'c'}
T_demo = 8
V_demo = 4

# Create realistic CTC output (model sees "abc")
# Alignment: _a_ab_bc -> abc
probs = np.array([
    [0.8, 0.1, 0.05, 0.05],  # t=0: mostly blank
    [0.1, 0.8, 0.05, 0.05],  # t=1: 'a'
    [0.7, 0.2, 0.05, 0.05],  # t=2: blank (separates 'a' from potential repeat)
    [0.2, 0.6, 0.15, 0.05],  # t=3: 'a' again (will be collapsed)
    [0.1, 0.1, 0.7, 0.1],    # t=4: 'b'
    [0.6, 0.1, 0.2, 0.1],    # t=5: blank
    [0.1, 0.05, 0.15, 0.7],  # t=6: 'c'
    [0.7, 0.1, 0.1, 0.1],    # t=7: blank
])

log_probs_demo = np.log(probs + 1e-10)

# Greedy decode
greedy_result, greedy_path = ctc_greedy_decode(log_probs_demo, blank=0)
print("Greedy path:", ''.join([ctc_vocab[t] for t in greedy_path]))
print("Greedy decoded:", ''.join([ctc_vocab[t] for t in greedy_result]))

# Beam search decode
beam_result, beam_sc, _ = ctc_beam_search(log_probs_demo, beam_width=5, blank=0)
print("Beam decoded:", ''.join([ctc_vocab[t] for t in beam_result]))

# Visualize the CTC output probabilities
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(T_demo)
colors = ['lightgray', 'steelblue', 'darkorange', 'forestgreen']
for v in range(V_demo):
    ax.bar(range(T_demo), probs[:, v], bottom=bottom, label=ctc_vocab[v],
           color=colors[v], edgecolor='black', alpha=0.8)
    bottom += probs[:, v]

ax.set_xlabel('Input Frame')
ax.set_ylabel('Probability')
ax.set_title('CTC Output Probabilities')
ax.legend()
ax.set_xticks(range(T_demo))
plt.tight_layout()
plt.show()

## Interview Questions & Answers

### "Viterbi -- walk me through it"
1. Build a trellis: rows = states, columns = timesteps
2. Initialize: V[0,k] = initial_prob[k] * emission_prob[k, obs_0]
3. Forward pass: V[t,j] = emission[j, obs_t] * max_i(V[t-1,i] * transition[i,j]). Store backpointer.
4. Termination: best_score = max_k(V[T,k])
5. Backtrack: follow backpointers from best ending state

Complexity: O(T * K^2). Works in log-space to avoid underflow.

### "CTC -- why is it needed?"
CTC solves the alignment problem when input length != output length. Instead of requiring frame-level labels, CTC:
- Adds a blank token to the vocabulary
- Marginalizes over all valid alignments (many-to-one mapping via collapsing repeats + removing blanks)
- Loss is computed using forward-backward algorithm (another DP)
- At inference: greedy (take argmax, collapse) or beam search (with prefix merging for correctness)

### "Beam search implementation and complexity"
- Maintain top-k hypotheses at each step
- For each hypothesis, compute scores for all V next tokens
- Keep top-k from the k*V candidates
- Complexity: O(T * k * V) per sequence (T = max length, k = beam width, V = vocab)
- In practice: k = 4-10 is common. Larger k gives diminishing returns.
- Variants: length normalization, coverage penalty, diverse beam search

## Where DP Appears in ML

| Application | Algorithm | What it solves |
|-------------|-----------|----------------|
| HMMs / CRFs | Viterbi | Best state sequence (structured prediction) |
| HMMs / CRFs | Forward-backward | Marginal probabilities for training |
| ASR / OCR | CTC | Alignment-free sequence labeling |
| MT / LLMs | Beam search | Approximate MAP decoding |
| Evaluation | Edit distance | WER, CER computation |
| Bioinformatics | Needleman-Wunsch | Sequence alignment |
| Optimal transport | Sinkhorn (iterative) | Alignment in representation learning |